# Teste 6 da PoC (v2): Baseline centralizado × STEP-GAN Federado com SMPC (modelo final do Teste 4 v5)

**Questão de pesquisa:** o STEP-GAN treinado de forma federada com agregação segura (SMPC) alcança desempenho
de detecção de fraude **não-inferior** ao mesmo modelo treinado de forma **centralizada** (todos os dados num
único servidor, sem privacidade)?

**Modelo:** as seções 0 a 4 deste notebook são **idênticas** às do Teste 4 (v5): STEP-GAN com geradores
condicionados por limiares e Discriminador graduado, SMPC em Z_p (p = 2^61−1), mesmos dados e hiperparâmetros.
Só a seção 5 (o experimento) é específica deste teste.

| Configuração | Papel no experimento |
|---|---|
| **Centralizado** | Baseline: a união dos dados dos 3 clientes num único modelo |
| **Federado + SMPC** | Modelo final (Teste 4 v5) |
| Local isolado | Limite inferior: cada banco treina sozinho (média dos 3 clientes) |

**Comparação justa (mesmo procedimento para todos):**
- mesmos pesos iniciais (D e os K geradores) por semente;
- mesmo orçamento: `NUM_RODADAS` blocos de `EPOCAS_LOCAIS` épocas (35 no total). No centralizado, cada bloco
  corresponde a uma rodada federada, com o otimizador reiniciado como no FL;
- **mesma seleção de modelo:** o checkpoint com melhor PR-AUC na validação, avaliado ao fim de cada bloco/rodada;
- limiar escolhido só na validação e métricas no teste.

**Estatística:** `SEMENTES` execuções pareadas; Δ = Federado+SMPC − Centralizado por semente (média, IC 95% t)
e IC 95% por bootstrap do Δ médio no teste. É **não-inferior** se o limite inferior do IC do Δ da
`METRICA_PRINCIPAL` for maior que −`MARGEM_NAO_INFERIORIDADE`. O resultado é reportado para ROC-AUC e PR-AUC.

In [ ]:
# -*- coding: utf-8 -*-
"""
Teste 6 da PoC (v2): Baseline centralizado do modelo final (seções 0-4 idênticas ao Teste 4 v5)
- STEP-GAN: K geradores condicionados por limiares (degraus da saída do Discriminador)
- Aprendizado Federado (3 silos) com agregação SMPC modular (p = 2^61 - 1)
- Avaliação com várias sementes (média ± desvio-padrão)
"""

import os
import copy
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             precision_recall_curve)

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
except ImportError:
    raise ImportError("PyTorch não está instalado. Execute: pip install torch")

# =====================================================================
# --- 0. CONFIGURAÇÃO ---
# =====================================================================
SEMENTES = [42, 43, 44, 45, 46]    # 5 a 10 sementes (comentário 1)
NUM_CLIENTES = 3
NUM_RODADAS = 5                     # Rodadas de comunicação do FL
EPOCAS_LOCAIS = 7                   # 5 x 7 = 35 épocas, mesmo total da versão anterior

# Hiperparâmetros do Teste 4
HP_BATCH_SIZE = 512
HP_LR_G = 0.0002
HP_LR_D = 0.0001
HP_LATENT_DIM = 100
HP_WEIGHT_DECAY = 1e-4
PESO_FRAUDE = 15.0                  # Loss ponderada para fraudes reais
ALVO_NORMAL = 0.9                   # Label smoothing: normal real
ALVO_ANOMALIA = 0.1                 # Fraude real e amostras geradas
PRECISAO_MINIMA = 0.70              # Limiar: maior recall com precisão >= 70% (na validação)
RAZAO_NORMAIS_POR_FRAUDE = 200      # Amostragem: 200 normais para cada fraude

# STEP-GAN: um gerador por degrau da saída do Discriminador (comentário 3)
# Degraus entre ALVO_ANOMALIA (0.1) e ALVO_NORMAL (0.9): pseudo-anomalias de "longe" a "perto" da fronteira
DEGRAUS = [(0.10, 0.35), (0.35, 0.60), (0.60, 0.85)]
DIM_CONDICAO = 16
# Discriminador graduado: a amostra de G_k(z, τ) recebe alvo (τ - MARGEM_GRADUADA) no Discriminador, em vez de 0.1.
# Assim D e G_k concordam sobre o degrau e o condicionamento passa a ser efetivo. Com todas as amostras geradas
# rotuladas como 0.1 (versão anterior), o condicionamento só funcionava para τ <= ~0.5 ou o desempenho caía.
# Ablação (Credit Card, semente 42): margem 0.0 -> PR-AUC 0.855 e correlação τ×D ~0.97; margem 0.15 -> PR-AUC 0.668.
DISCRIMINADOR_GRADUADO = True
MARGEM_GRADUADA = 0.0

# SMPC (comentário 2)
SMPC_MODULO = 2**61 - 1             # Primo de Mersenne; partes em [0, p) e somas de 2 partes < 2^62 cabem em int64
SMPC_BITS_PRECISAO = 24             # Resolução do ponto fixo: 2^-24 ≈ 6e-8
SMPC_ESCALA = 2 ** SMPC_BITS_PRECISAO
SMPC_LIMITE_SOMA = (SMPC_MODULO // 2) / SMPC_ESCALA   # |Σ| representável ≈ 6.9e10

DATASETS = ['paysim', 'creditcard']
CAMINHOS = {
    'creditcard': [
        '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/creditcard/creditcard.csv',
        'base_de_dados/creditcard.zip',        # Executando de dentro de PoC/
        'PoC/base_de_dados/creditcard.zip',    # Executando da raiz do repositório
    ],
    'paysim': [
        '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/paysim.csv',
    ],
}


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[SETUP] Dispositivo PyTorch: {device}")


def sincronizar():
    """Garante que o cronômetro só pare depois que a GPU terminar (CUDA é assíncrono)."""
    if device.type == 'cuda':
        torch.cuda.synchronize()


try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Fora do Google Colab: usando apenas caminhos locais.")

In [ ]:
# =====================================================================
# --- 1. PROTOCOLO SMPC: COMPARTILHAMENTO ADITIVO EM Z_p (comentário 2) ---
# =====================================================================

def codificar_ponto_fixo(v):
    """
    float64 -> Z_p. Arredonda para o inteiro mais próximo de v * 2^24 e representa negativos como p - |x|.
    Ex.: -1.5 -> round(-1.5 * 2^24) = -25165824 -> p - 25165824.
    """
    return torch.remainder(torch.round(v * SMPC_ESCALA).to(torch.int64), SMPC_MODULO)


def decodificar_ponto_fixo(x):
    """Z_p -> float64. Valores acima de p/2 são interpretados como negativos."""
    x = torch.where(x > SMPC_MODULO // 2, x - SMPC_MODULO, x)
    return x.to(torch.float64) / SMPC_ESCALA


def mascaras_csprng(forma):
    """Inteiros uniformes em [0, p) gerados com os.urandom (CSPRNG do sistema operacional)."""
    n = int(np.prod(forma))
    brutos = np.frombuffer(os.urandom(8 * n), dtype=np.uint64) & np.uint64(SMPC_MODULO)  # 61 bits
    brutos = brutos % np.uint64(SMPC_MODULO)                                             # mapeia p -> 0
    return torch.from_numpy(brutos.astype(np.int64)).view(forma)


def soma_modular(tensores):
    """Soma reduzindo mod p a cada passo: cada parcela < 2^61, então a soma de duas < 2^62 nunca estoura o int64."""
    total = torch.zeros_like(tensores[0])
    for t in tensores:
        total = torch.remainder(total + t, SMPC_MODULO)
    return total


def gerar_partes(x_int, n):
    """Divide x (já em Z_p) em n partes aleatórias com Σ partes ≡ x (mod p). Cada parte isolada é uniforme em Z_p."""
    mascaras = [mascaras_csprng(x_int.shape) for _ in range(n - 1)]
    ultima = x_int
    for m in mascaras:
        ultima = torch.remainder(ultima - m, SMPC_MODULO)  # diferença em (-p, p): sem estouro
    return mascaras + [ultima]


def verificar_faixa(v, n):
    """
    Checagem de estouro feita por cada cliente antes de compartilhar: se todo |v_k| <= LIMITE / n,
    então |Σ v_k| < p / 2 e a decodificação de negativos continua correta.
    """
    maximo = v.abs().max().item()
    if maximo > SMPC_LIMITE_SOMA / n:
        raise ValueError(f"Estouro do módulo SMPC: |valor| = {maximo:.3e} > {SMPC_LIMITE_SOMA / n:.3e}. "
                         f"Aumente SMPC_MODULO ou reduza SMPC_BITS_PRECISAO.")


def achatar(estado):
    """Concatena todos os tensores de ponto flutuante do estado (pesos e buffers do BatchNorm) em float64."""
    return torch.cat([t.detach().cpu().to(torch.float64).flatten()
                      for t in estado.values() if t.is_floating_point()])


def desachatar(vetor, estado_ref, estados_clientes):
    novo, pos = {}, 0
    for chave, t in estado_ref.items():
        if t.is_floating_point():
            novo[chave] = vetor[pos:pos + t.numel()].view(t.shape).to(t.dtype)
            pos += t.numel()
        else:
            # num_batches_tracked do BatchNorm: contador inteiro sem informação dos dados (não é usado com momentum)
            novo[chave] = torch.stack([e[chave] for e in estados_clientes]).max()
    return novo


def agregar_smpc(estados_clientes, pesos):
    """
    FedAvg ponderado via SMPC com n nós (os próprios clientes):
      1. Cliente k: pondera (w_k * n_k/N), verifica a faixa, codifica em Z_p e gera n partes; a parte j vai para o nó j.
      2. Nó j: soma (mod p) apenas as partes que recebeu, que são individualmente uniformes (não revelam nada).
      3. Servidor: soma (mod p) as n somas parciais e decodifica. Ele só conhece o agregado Σ_k w_k n_k/N.
    """
    n = len(estados_clientes)
    t0 = time.perf_counter()
    caixas = [[] for _ in range(n)]                      # caixas[j]: partes recebidas pelo nó j
    for estado, peso in zip(estados_clientes, pesos):
        v = achatar(estado) * peso
        verificar_faixa(v, n)
        for j, parte in enumerate(gerar_partes(codificar_ponto_fixo(v), n)):
            caixas[j].append(parte)
    somas_parciais = [soma_modular(caixas[j]) for j in range(n)]
    agregado = decodificar_ponto_fixo(soma_modular(somas_parciais))
    t_smpc = time.perf_counter() - t0

    referencia = sum(achatar(e) * p for e, p in zip(estados_clientes, pesos))   # FedAvg em claro
    erro_max = (agregado - referencia).abs().max().item()
    bytes_cliente = n * agregado.numel() * 8           # (n-1) partes aos outros nós + 1 soma parcial
    return desachatar(agregado, estados_clientes[0], estados_clientes), dict(
        t_smpc=t_smpc, erro_max=erro_max, bytes_cliente=bytes_cliente)


def autoteste_smpc():
    print("--- Autoteste do SMPC ---")
    n, pesos = 3, [1 / 3] * 3
    valores = [torch.tensor([-1.5, 30.0, -30.0, 1595.5, -2.0e7, 7e-8, 0.123456789], dtype=torch.float64) * f
               for f in (1.0, 0.5, -0.25)]
    esperado = sum(v * p for v, p in zip(valores, pesos))
    caixas = [[] for _ in range(n)]
    for v, p in zip(valores, pesos):
        for j, parte in enumerate(gerar_partes(codificar_ponto_fixo(v * p), n)):
            caixas[j].append(parte)
    obtido = decodificar_ponto_fixo(soma_modular([soma_modular(c) for c in caixas]))
    erro = (obtido - esperado).abs().max().item()
    tolerancia = n * 0.5 / SMPC_ESCALA
    assert erro <= tolerancia, f"Erro de reconstrução {erro} > {tolerancia}"
    print(f"  Reconstrução com negativos e valores grandes: erro máx. {erro:.2e} (tolerância {tolerancia:.2e}) OK")

    parte_isolada = decodificar_ponto_fixo(caixas[0][0])
    print(f"  Uma parte isolada decodificada parece ruído: {parte_isolada[:3].tolist()}")

    try:
        verificar_faixa(torch.tensor([SMPC_LIMITE_SOMA], dtype=torch.float64), n)
        raise AssertionError("A verificação de estouro deveria ter falhado")
    except ValueError:
        print(f"  Estouro detectado antes de compartilhar (limite por cliente {SMPC_LIMITE_SOMA / n:.2e}) OK")

    p_antigo, escala_antiga = 2147483647, 2 ** 24
    soma_antiga = 3 * round(30.0 * escala_antiga) % p_antigo
    soma_antiga = soma_antiga - p_antigo if soma_antiga > p_antigo // 2 else soma_antiga
    print(f"  Configuração antiga (p = 2^31-1): 3 clientes com peso 30 -> média {soma_antiga / escala_antiga / 3:+.2f} "
          f"(esperado +30.00): estouro silencioso corrigido.")


autoteste_smpc()

In [ ]:
# =====================================================================
# --- 2. DADOS (engenharia de features do Teste 4, sem data leakage) ---
# =====================================================================

def ler_base(nome):
    for caminho in CAMINHOS[nome]:
        if os.path.exists(caminho):
            print(f"Carregando '{caminho}'...")
            return pd.read_csv(caminho)
    return None


def amostrar(df, alvo):
    """Mantém todas as fraudes e RAZAO_NORMAIS_POR_FRAUDE vezes mais transações normais."""
    fraudes = df[df[alvo] == 1]
    normais = df[df[alvo] == 0]
    normais = normais.sample(n=min(len(fraudes) * RAZAO_NORMAIS_POR_FRAUDE, len(normais)), random_state=42)
    return pd.concat([normais, fraudes]).sample(frac=1, random_state=42).reset_index(drop=True)


def preparar_creditcard(df):
    df = amostrar(df.dropna(subset=['Class']), 'Class')
    df = df.drop(columns=['Time'], errors='ignore')      # Contador monotônico: ruído para o GAN
    df['amount_log'] = np.log1p(df['Amount'].clip(lower=0))
    return df.drop(columns=['Amount']), 'Class'


def preparar_paysim(df):
    df = amostrar(df.dropna(subset=['isFraud']), 'isFraud')
    df['orig_balance_zero'] = (df['oldbalanceOrg'] == 0).astype(int)
    df['dest_balance_zero'] = (df['oldbalanceDest'] == 0).astype(int)
    for col in ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']:
        df[col + '_log'] = np.log1p(df[col].clip(lower=0))
    df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
    df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
    df = df.drop(columns=['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'])
    df = df.drop(columns=[c for c in ['nameOrig', 'nameDest', 'isFlaggedFraud'] if c in df.columns])
    return pd.get_dummies(df, columns=['type'], drop_first=True, dtype=float), 'isFraud'


PREPARADORES = {'creditcard': preparar_creditcard, 'paysim': preparar_paysim}


def carregar_dados(nome):
    df = ler_base(nome)
    if df is None:
        return None
    df, alvo = PREPARADORES[nome](df)
    X = df.drop(columns=[alvo]).values.astype(np.float32)
    y = df[alvo].values.astype(np.int64)

    # Split 60/20/20 estratificado ANTES de escalar
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    # MinMaxScaler(-1, 1) alinhado ao Tanh dos geradores, ajustado SÓ no treino
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    # Silos bancários: fatias do treino
    fatias = np.array_split(np.arange(len(X_train)), NUM_CLIENTES)
    clientes = [(X_train[i], y_train[i]) for i in fatias]
    pesos_fedavg = [len(i) / len(X_train) for i in fatias]

    print(f"[DADOS] Features: {X_train.shape[1]} | Treino: {len(X_train)} (fraudes: {int(y_train.sum())}) "
          f"| Validação: {len(X_val)} (fraudes: {int(y_val.sum())}) | Teste: {len(X_test)} (fraudes: {int(y_test.sum())})")
    return dict(nome=nome, input_dim=X_train.shape[1], clientes=clientes, pesos_fedavg=pesos_fedavg,
                X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test)

In [ ]:
# =====================================================================
# --- 3. STEP-GAN: GERADORES CONDICIONADOS POR LIMIARES (comentário 3) ---
# =====================================================================

class GeradorCondicional(nn.Module):
    """G_k(z, τ): gera pseudo-anomalias cuja saída no Discriminador deve ficar em τ (limiar do degrau k)."""

    def __init__(self, input_dim, latent_dim=HP_LATENT_DIM):
        super().__init__()
        self.condicao = nn.Sequential(nn.Linear(1, DIM_CONDICAO), nn.LeakyReLU(0.2))
        self.net = nn.Sequential(
            nn.Linear(latent_dim + DIM_CONDICAO, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Linear(512, input_dim), nn.Tanh()
        )

    def forward(self, z, tau):
        return self.net(torch.cat([z, self.condicao(tau)], dim=1))


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


def amostrar_tau(k, n):
    """Limiar τ ~ U(degrau k): a condição do gerador k."""
    minimo, maximo = DEGRAUS[k]
    return torch.rand(n, 1, device=device) * (maximo - minimo) + minimo


def custom_bce_loss(pred, target, real_labels_flag):
    """BCE com peso condicional: erro em fraude real pesa PESO_FRAUDE vezes mais."""
    loss = nn.BCELoss(reduction='none')(pred, target)
    weights = torch.ones_like(target)
    weights[(target < 0.5) & real_labels_flag] = PESO_FRAUDE
    return (loss * weights).mean()


def novo_estado_global(input_dim):
    return {'D': copy.deepcopy(Discriminator(input_dim).state_dict()),
            'G': [copy.deepcopy(GeradorCondicional(input_dim).state_dict()) for _ in DEGRAUS]}


def treinar_local(estado_global, dados_X, dados_y, input_dim, semente):
    """Treino local do STEP-GAN a partir do modelo global (D + K geradores)."""
    torch.manual_seed(semente)
    K = len(DEGRAUS)
    D = Discriminator(input_dim).to(device)
    D.load_state_dict(estado_global['D'])
    Gs = []
    for estado_G in estado_global['G']:
        G = GeradorCondicional(input_dim).to(device)
        G.load_state_dict(estado_G)
        Gs.append(G)
    D.train()
    for G in Gs:
        G.train()

    opt_D = optim.Adam(D.parameters(), lr=HP_LR_D, betas=(0.5, 0.999), weight_decay=HP_WEIGHT_DECAY)
    opts_G = [optim.Adam(G.parameters(), lr=HP_LR_G, betas=(0.5, 0.999), weight_decay=HP_WEIGHT_DECAY) for G in Gs]
    loader = DataLoader(TensorDataset(torch.from_numpy(dados_X), torch.from_numpy(dados_y).float()),
                        batch_size=HP_BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(semente))

    for _ in range(EPOCAS_LOCAIS):
        for i, (real_data, real_y) in enumerate(loader):
            real_data, real_y = real_data.to(device), real_y.to(device)
            batch_s = real_data.size(0)
            n_k = max(batch_s // K, 2)                   # BatchNorm exige pelo menos 2 amostras
            if batch_s < 2:
                continue

            real_labels = torch.where(real_y.unsqueeze(1) == 0,
                                      torch.tensor(ALVO_NORMAL, device=device),
                                      torch.tensor(ALVO_ANOMALIA, device=device))
            taus = [amostrar_tau(k, n_k) for k in range(K)]
            fakes = [G(torch.randn(n_k, HP_LATENT_DIM, device=device), tau) for G, tau in zip(Gs, taus)]

            # --- Discriminador graduado: normal real -> 0.9; fraude real -> 0.1; amostra do degrau τ -> τ ---
            opt_D.zero_grad()
            loss_real = custom_bce_loss(D(real_data), real_labels, torch.ones_like(real_labels, dtype=torch.bool))
            fake_cat = torch.cat([f.detach() for f in fakes])
            if DISCRIMINADOR_GRADUADO:
                fake_labels = torch.cat(taus).sub(MARGEM_GRADUADA).clamp(min=ALVO_ANOMALIA)
            else:
                fake_labels = torch.full((fake_cat.size(0), 1), ALVO_ANOMALIA, device=device)
            loss_fake = custom_bce_loss(D(fake_cat), fake_labels, torch.zeros_like(fake_labels, dtype=torch.bool))
            (loss_real + loss_fake).backward()
            torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
            opt_D.step()

            # --- Geradores (a cada 2 passos do D): condicionamento D(G_k(z, τ)) -> τ ---
            if i % 2 == 0:
                for G, opt_G, fake, tau in zip(Gs, opts_G, fakes, taus):
                    opt_G.zero_grad()
                    loss_G = nn.BCELoss()(D(fake), tau)
                    loss_G.backward()
                    torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
                    opt_G.step()

    return {'D': {k: v.detach().cpu().clone() for k, v in D.state_dict().items()},
            'G': [{k: v.detach().cpu().clone() for k, v in G.state_dict().items()} for G in Gs]}

In [ ]:
# =====================================================================
# --- 4. APRENDIZADO FEDERADO + AVALIAÇÃO ---
# =====================================================================

def scores_anomalia(estado_D, input_dim, X, tamanho_lote=65536):
    D = Discriminator(input_dim).to(device)
    D.load_state_dict(estado_D)
    D.eval()
    saida = []
    with torch.no_grad():
        for i in range(0, len(X), tamanho_lote):
            saida.append(D(torch.from_numpy(X[i:i + tamanho_lote]).to(device)).cpu().numpy().ravel())
    return 1.0 - np.concatenate(saida)        # D ≈ 0.9 normal, ≈ 0.1 anomalia


def escolher_limiar(y_val, s_val):
    """Maior recall com precisão >= PRECISAO_MINIMA na VALIDAÇÃO; se inalcançável, máximo F1."""
    precisions, recalls, thresholds = precision_recall_curve(y_val, s_val)
    validos = np.where(precisions[:-1] >= PRECISAO_MINIMA)[0]
    if len(validos) > 0:
        return thresholds[validos[np.argmax(recalls[validos])]]
    f1 = np.divide(2 * precisions * recalls, precisions + recalls,
                   out=np.zeros_like(precisions), where=(precisions + recalls) != 0)
    return thresholds[np.argmax(f1[:-1])]


def executar_federado(dados, semente):
    """FL com NUM_RODADAS rodadas. D e os K geradores são agregados via SMPC. Retorna o global da melhor rodada."""
    input_dim = dados['input_dim']
    seed_everything(semente)
    estado_global = novo_estado_global(input_dim)
    historico, melhor = [], None

    for rodada in range(1, NUM_RODADAS + 1):
        sincronizar()
        t0 = time.perf_counter()
        locais = [treinar_local(estado_global, X_k, y_k, input_dim, semente * 1000 + rodada * 10 + k)
                  for k, (X_k, y_k) in enumerate(dados['clientes'])]
        sincronizar()
        t_treino = time.perf_counter() - t0

        # Um pacote por cliente com D e todos os geradores: uma única agregação SMPC por rodada
        pacotes = [{**{f'D.{c}': t for c, t in l['D'].items()},
                    **{f'G{k}.{c}': t for k, g in enumerate(l['G']) for c, t in g.items()}} for l in locais]
        agregado, custo = agregar_smpc(pacotes, dados['pesos_fedavg'])
        estado_global = {'D': {c[2:]: t for c, t in agregado.items() if c.startswith('D.')},
                         'G': [{c[len(f'G{k}.'):]: t for c, t in agregado.items() if c.startswith(f'G{k}.')}
                               for k in range(len(DEGRAUS))]}

        pr_auc_val = average_precision_score(dados['y_val'], scores_anomalia(estado_global['D'], input_dim, dados['X_val']))
        historico.append(dict(rodada=rodada, pr_auc_val=pr_auc_val, t_treino=t_treino, **custo))
        print(f"  Rodada {rodada}/{NUM_RODADAS} | PR-AUC val {pr_auc_val:.4f} | treino {t_treino:.1f}s "
              f"| SMPC {1000 * custo['t_smpc']:.0f} ms (erro máx. vs claro {custo['erro_max']:.1e})")
        if melhor is None or pr_auc_val > melhor[0]:
            melhor = (pr_auc_val, rodada, copy.deepcopy(estado_global))

    return melhor[2], melhor[1], pd.DataFrame(historico)


def avaliar(estado_D, dados):
    input_dim = dados['input_dim']
    limiar = escolher_limiar(dados['y_val'], scores_anomalia(estado_D, input_dim, dados['X_val']))
    sincronizar()
    t0 = time.perf_counter()
    s = scores_anomalia(estado_D, input_dim, dados['X_test'])
    sincronizar()
    t_inf = time.perf_counter() - t0
    y, y_pred = dados['y_test'], (s >= limiar).astype(int)
    return dict(scores=s, y_pred=y_pred, limiar=limiar, **{
        'ROC-AUC': roc_auc_score(y, s), 'PR-AUC': average_precision_score(y, s),
        'Precisão': precision_score(y, y_pred, zero_division=0), 'Recall': recall_score(y, y_pred, zero_division=0),
        'F1': f1_score(y, y_pred, zero_division=0), 'Acurácia': accuracy_score(y, y_pred),
        'TPS': len(y) / t_inf if t_inf > 0 else float('nan')})


def aderencia_condicionamento(estado_global, input_dim, n=2000):
    """Mede se cada gerador respeita seu limiar: compara τ pedido com D(G_k(z, τ)) obtido."""
    D = Discriminator(input_dim).to(device)
    D.load_state_dict(estado_global['D'])
    D.eval()
    resultado = []
    with torch.no_grad():
        for k, estado_G in enumerate(estado_global['G']):
            G = GeradorCondicional(input_dim).to(device)
            G.load_state_dict(estado_G)
            G.eval()
            tau = amostrar_tau(k, n)
            saida = D(G(torch.randn(n, HP_LATENT_DIM, device=device), tau))
            resultado.append(pd.DataFrame({'gerador': f'G{k + 1} {DEGRAUS[k]}',
                                           'tau': tau.cpu().numpy().ravel(), 'D': saida.cpu().numpy().ravel()}))
    return pd.concat(resultado, ignore_index=True)

In [ ]:
# =====================================================================
# --- 5. EXPERIMENTO: CENTRALIZADO × FEDERADO + SMPC (Teste 6) ---
# =====================================================================
from sklearn.metrics import roc_curve

METRICA_PRINCIPAL = 'ROC-AUC'       # Métrica do teste de não-inferioridade ('ROC-AUC' ou 'PR-AUC')
MARGEM_NAO_INFERIORIDADE = 0.01     # Perda máxima aceitável do federado em relação ao centralizado
N_BOOTSTRAP = 300                   # Reamostragens do teste para o IC 95% do Δ médio

METRICAS = ['ROC-AUC', 'PR-AUC', 'Precisão', 'Recall', 'F1', 'Acurácia']
CENTRAL, FEDERADO, LOCAL = 'Centralizado', 'Federado + SMPC', 'Local isolado'
CONFIGS = [CENTRAL, FEDERADO, LOCAL]
CORES = {CENTRAL: '#1f77b4', FEDERADO: '#d62728', LOCAL: '#7f7f7f'}


def treinar_em_blocos(dados, X, y, semente, id_semente_bloco):
    """
    Treino SEM federação com o mesmo procedimento do FL: parte dos mesmos pesos iniciais, treina NUM_RODADAS blocos
    de EPOCAS_LOCAIS épocas e guarda o checkpoint com melhor PR-AUC na validação.
    """
    input_dim = dados['input_dim']
    seed_everything(semente)
    estado = novo_estado_global(input_dim)          # Mesma inicialização de executar_federado()
    historico, melhor, t_total = [], None, 0.0
    for bloco in range(1, NUM_RODADAS + 1):
        sincronizar()
        t0 = time.perf_counter()
        estado = treinar_local(estado, X, y, input_dim, semente * 1000 + bloco * 10 + id_semente_bloco)
        sincronizar()
        t_total += time.perf_counter() - t0
        pr_auc_val = average_precision_score(dados['y_val'], scores_anomalia(estado['D'], input_dim, dados['X_val']))
        historico.append(dict(rodada=bloco, pr_auc_val=pr_auc_val))
        if melhor is None or pr_auc_val > melhor[0]:
            melhor = (pr_auc_val, bloco, copy.deepcopy(estado))
    return melhor[2], melhor[1], pd.DataFrame(historico), t_total


def bootstrap_delta(y, pares_scores, n_boot, semente):
    """IC 95% do Δ médio entre sementes (federado − centralizado), reamostrando o conjunto de teste."""
    rng = np.random.default_rng(semente)
    deltas = {'ROC-AUC': [], 'PR-AUC': []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        for m, fn in [('ROC-AUC', roc_auc_score), ('PR-AUC', average_precision_score)]:
            deltas[m].append(np.mean([fn(y[idx], f[idx]) - fn(y[idx], c[idx]) for c, f in pares_scores]))
    return {m: np.array(v) for m, v in deltas.items()}


resultados_gerais = {}

for nome_base in DATASETS:
    print("\n" + "=" * 78)
    print(f" BASE: {nome_base.upper()} | {len(SEMENTES)} sementes | {NUM_RODADAS} x {EPOCAS_LOCAIS} épocas "
          f"| {len(DEGRAUS)} geradores")
    print("=" * 78)
    dados = carregar_dados(nome_base)
    if dados is None:
        print(f"Aviso: base '{nome_base}' não encontrada em {CAMINHOS[nome_base]}. Pulando.")
        continue
    X_central = np.concatenate([c[0] for c in dados['clientes']])   # União exata dos dados dos clientes
    y_central = np.concatenate([c[1] for c in dados['clientes']])

    linhas, tempos, pares_scores, curvas, exemplo = [], [], [], [], None
    for semente in SEMENTES:
        print(f"\n--- Semente {semente} ---")
        # (a) Centralizado
        est_c, bloco_c, hist_c, t_c = treinar_em_blocos(dados, X_central, y_central, semente, id_semente_bloco=9)
        print(f"[Centralizado] treino {t_c:.1f}s | melhor bloco {bloco_c} | PR-AUC val {hist_c['pr_auc_val'].max():.4f}")

        # (b) Federado + SMPC (modelo final)
        est_f, rodada_f, hist_f = executar_federado(dados, semente)
        t_f = hist_f['t_treino'].sum()

        # (c) Local isolado: cada cliente com as mesmas sementes que usa no federado
        avs_locais = []
        for k, (X_k, y_k) in enumerate(dados['clientes']):
            est_k, _, _, _ = treinar_em_blocos(dados, X_k, y_k, semente, id_semente_bloco=k)
            avs_locais.append(avaliar(est_k['D'], dados))
        print(f"[Local isolado] PR-AUC teste por cliente: {[round(a['PR-AUC'], 4) for a in avs_locais]}")

        av = {CENTRAL: avaliar(est_c['D'], dados), FEDERADO: avaliar(est_f['D'], dados),
              LOCAL: {m: np.mean([a[m] for a in avs_locais]) for m in METRICAS}}
        for cfg in CONFIGS:
            linhas.append({'Semente': semente, 'Configuração': cfg, **{m: av[cfg][m] for m in METRICAS}})
            print(f"  {cfg:16s} ROC-AUC {av[cfg]['ROC-AUC']:.4f} | PR-AUC {av[cfg]['PR-AUC']:.4f} | F1 {av[cfg]['F1']:.4f}")
        tempos.append({'Semente': semente, 'Centralizado': t_c, 'Federado (sequencial)': t_f,
                       'Federado (paralelo, estimado)': t_f / NUM_CLIENTES,
                       'SMPC total': hist_f['t_smpc'].sum(), 'Erro máx. SMPC': hist_f['erro_max'].max(),
                       'Bloco escolhido (central)': bloco_c, 'Rodada escolhida (federado)': rodada_f})
        pares_scores.append((av[CENTRAL]['scores'], av[FEDERADO]['scores']))
        curvas.append((hist_c.assign(semente=semente, cfg=CENTRAL), hist_f[['rodada', 'pr_auc_val']].assign(semente=semente, cfg=FEDERADO)))
        if exemplo is None:
            exemplo = {cfg: av[cfg]['scores'] for cfg in [CENTRAL, FEDERADO]}

    df = pd.DataFrame(linhas)
    df_tempos = pd.DataFrame(tempos).set_index('Semente')
    df_curvas = pd.concat([pd.concat(c) for c in curvas], ignore_index=True)

    # ----- Resumo -----
    resumo = df.groupby('Configuração')[METRICAS].agg(['mean', 'std']).reindex(CONFIGS)
    print(f"\n>> MÉTRICAS NO TESTE: média ± desvio-padrão em {len(SEMENTES)} sementes")
    print(pd.DataFrame({m: [f"{resumo.loc[c, (m, 'mean')]:.4f} ± {np.nan_to_num(resumo.loc[c, (m, 'std')]):.4f}"
                            for c in CONFIGS] for m in METRICAS}, index=CONFIGS).to_string())

    piv = df.pivot(index='Semente', columns='Configuração')
    deltas = {m: (piv[(m, FEDERADO)] - piv[(m, CENTRAL)]).values for m in METRICAS}
    print(f"\n>> Δ PAREADO POR SEMENTE ({FEDERADO} − {CENTRAL}):")
    for m in METRICAS:
        d = deltas[m]
        meia = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else 0.0
        print(f"  {m:9s}: {d.mean():+.4f}  (IC95% entre sementes: [{d.mean() - meia:+.4f}, {d.mean() + meia:+.4f}])")

    boot = bootstrap_delta(dados['y_test'], pares_scores, N_BOOTSTRAP, SEMENTES[0])
    ic = {m: np.percentile(v, [2.5, 97.5]) for m, v in boot.items()}
    print(f"\n>> IC 95% POR BOOTSTRAP DO Δ MÉDIO ({N_BOOTSTRAP} reamostragens do teste):")
    for m in ['ROC-AUC', 'PR-AUC']:
        veredito = 'não-inferior' if ic[m][0] > -MARGEM_NAO_INFERIORIDADE else 'não-inferioridade NÃO demonstrada'
        print(f"  Δ{m}: médio {deltas[m].mean():+.4f} | IC95% [{ic[m][0]:+.4f}, {ic[m][1]:+.4f}] -> {veredito} "
              f"(margem {MARGEM_NAO_INFERIORIDADE})")

    nao_inferior = ic[METRICA_PRINCIPAL][0] > -MARGEM_NAO_INFERIORIDADE
    print(f"\n>> RESPOSTA À QUESTÃO DE PESQUISA (métrica principal: {METRICA_PRINCIPAL}):")
    if nao_inferior:
        print(f"  O STEP-GAN Federado + SMPC é NÃO-INFERIOR ao baseline centralizado: limite inferior do IC95% "
              f"{ic[METRICA_PRINCIPAL][0]:+.4f} > {-MARGEM_NAO_INFERIORIDADE}.")
    else:
        print(f"  NÃO foi possível demonstrar não-inferioridade: limite inferior do IC95% "
              f"{ic[METRICA_PRINCIPAL][0]:+.4f} <= {-MARGEM_NAO_INFERIORIDADE}.")
    ganho = (piv[(METRICA_PRINCIPAL, FEDERADO)] - piv[(METRICA_PRINCIPAL, LOCAL)]).mean()
    print(f"  Ganho da colaboração ({FEDERADO} − {LOCAL}): {ganho:+.4f} {METRICA_PRINCIPAL}")
    print(f"  SMPC: erro máx. vs FedAvg em claro {df_tempos['Erro máx. SMPC'].max():.1e}")

    print("\n>> TEMPOS (s) E CHECKPOINTS ESCOLHIDOS POR SEMENTE:")
    print(df_tempos.to_string(formatters={'Erro máx. SMPC': lambda x: f"{x:.1e}"}, float_format=lambda x: f"{x:.2f}"))

    arquivo = f'resultados_teste6_{nome_base}.csv'
    df.to_csv(arquivo, index=False)
    print(f"Resultados por semente salvos em '{arquivo}'.")
    resultados_gerais[nome_base] = dict(metricas=df, tempos=df_tempos, ic_bootstrap=ic, nao_inferior=nao_inferior)

    # ----- Visualizações -----
    fig, axes = plt.subplots(2, 3, figsize=(22, 11))
    fig.suptitle(f"Teste 6 (v2): Centralizado × Federado + SMPC (STEP-GAN v5, {nome_base}, {len(SEMENTES)} sementes)",
                 fontsize=14, fontweight='bold')

    # 1. Métricas
    ax = axes[0, 0]
    largura = 0.8 / len(CONFIGS)
    x = np.arange(len(METRICAS))
    for i, cfg in enumerate(CONFIGS):
        ax.bar(x + (i - (len(CONFIGS) - 1) / 2) * largura, [resumo.loc[cfg, (m, 'mean')] for m in METRICAS], largura,
               yerr=[np.nan_to_num(resumo.loc[cfg, (m, 'std')]) for m in METRICAS], capsize=3,
               color=CORES[cfg], label=cfg)
    ax.set_xticks(x)
    ax.set_xticklabels(METRICAS)
    ax.set_ylim(0, 1.2)
    ax.set_title('Métricas no teste (média ± desvio)', fontweight='bold')
    ax.legend(loc='upper center', ncol=3)

    # 2. Curvas ROC (primeira semente)
    ax = axes[0, 1]
    for cfg in [CENTRAL, FEDERADO]:
        fpr, tpr, _ = roc_curve(dados['y_test'], exemplo[cfg])
        ax.plot(fpr, tpr, '-' if cfg == CENTRAL else '--', color=CORES[cfg],
                label=f"{cfg} (AUC={roc_auc_score(dados['y_test'], exemplo[cfg]):.4f})")
    ax.plot([0, 1], [0, 1], ':', color='gray')
    ax.set_title(f'Curva ROC no teste (semente {SEMENTES[0]})', fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(loc='lower right')

    # 3. Convergência: PR-AUC na validação por bloco/rodada
    ax = axes[0, 2]
    for cfg in [CENTRAL, FEDERADO]:
        sub = df_curvas[df_curvas['cfg'] == cfg]
        for _, g in sub.groupby('semente'):
            ax.plot(g['rodada'], g['pr_auc_val'], color=CORES[cfg], alpha=0.25)
        media = sub.groupby('rodada')['pr_auc_val'].mean()
        ax.plot(media.index, media.values, color=CORES[cfg], lw=2.5, marker='o', label=f'{cfg} (média)')
    ax.set_xlabel(f'Rodada / bloco de {EPOCAS_LOCAIS} épocas')
    ax.set_title('PR-AUC na validação', fontweight='bold')
    ax.legend()

    # 4. Δ por semente e IC de não-inferioridade
    ax = axes[1, 0]
    for i, m in enumerate(['ROC-AUC', 'PR-AUC', 'F1']):
        ax.scatter(np.full(len(deltas[m]), i), deltas[m], color=CORES[FEDERADO], zorder=3,
                   label='Δ por semente' if i == 0 else None)
        ax.plot([i - 0.2, i + 0.2], [deltas[m].mean()] * 2, color='black', lw=2, label='Δ médio' if i == 0 else None)
    for i, m in enumerate(['ROC-AUC', 'PR-AUC']):
        ax.errorbar(i + 0.3, np.mean(ic[m]), yerr=[[np.mean(ic[m]) - ic[m][0]], [ic[m][1] - np.mean(ic[m])]],
                    fmt='none', ecolor='#2ca02c', capsize=6, lw=2, label='IC95% bootstrap' if i == 0 else None)
    ax.axhline(0, color='gray', lw=1)
    ax.axhline(-MARGEM_NAO_INFERIORIDADE, color='#d62728', ls='--',
               label=f'Margem de não-inferioridade (−{MARGEM_NAO_INFERIORIDADE})')
    ax.set_xticks(range(3))
    ax.set_xticklabels(['ΔROC-AUC', 'ΔPR-AUC', 'ΔF1'])
    ax.set_title(f'{FEDERADO} − {CENTRAL}', fontweight='bold')
    ax.legend(loc='best', fontsize=9)

    # 5. Tempo de treino
    ax = axes[1, 1]
    media_t = df_tempos[['Centralizado', 'Federado (sequencial)', 'Federado (paralelo, estimado)']].mean()
    ax.bar(media_t.index, media_t.values, color=['#1f77b4', '#ff9896', '#d62728'])
    for i, v in enumerate(media_t.values):
        ax.text(i, v, f"{v:.1f}s", ha='center', va='bottom')
    ax.set_title(f"Tempo de treino (SMPC total: {df_tempos['SMPC total'].mean():.2f}s)", fontweight='bold')
    ax.set_ylabel('segundos')

    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.show()

print("\n[SUCESSO] Teste 6 (v2) concluído!")

## Nova base: Healthcare Fraud Detection

As células abaixo foram **adicionadas** ao Teste 6, sem alterar as anteriores. Elas repetem o experimento da
seção 5 (baseline centralizado × STEP-GAN Federado + SMPC) com a **mesma configuração** — modelo, hiperparâmetros, sementes, split, limiar e métricas — na base
`healthcare_fraud_detection.csv`, carregada direto do Google Drive
(`/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/`). O rótulo de fraude é a coluna `Is_Fraud`.

1. A primeira célula registra a base (caminho e engenharia de features) no pipeline do teste.
2. A segunda executa o experimento na base healthcare.
3. A terceira compara os resultados com as outras bases executadas neste notebook.

**Engenharia de features da base healthcare** (a mesma nos seis testes):
- removidos `Provider_ID` e `Claim_ID` (identificadores, como `nameOrig`/`nameDest` na PaySim) e `Claim_Submission_Date` (carimbo de tempo, como `Time` na Credit Card);
- `claim_amount_log` e `approved_amount_log` (log dos valores, como `amount_log`) e `approved_ratio` = valor aprovado ÷ valor solicitado (análoga ao `errorBalance` da PaySim);
- One-Hot das categóricas (`Patient_Gender`, `Diagnosis_Code`, `Procedure_Code`, `Insurance_Type`, `Provider_Specialty`, `Patient_State`, `Claim_Status`, `Visit_Type`), com ausentes na categoria `Desconhecido`;
- mediana para numéricas ausentes (`Prior_Visits_12m`).

In [ ]:
# =====================================================================
# --- NOVA BASE: Healthcare Fraud Detection (mesma configuração do teste) ---
# =====================================================================
CAMINHOS['healthcare'] = [
    '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/healthcare_fraud_detection.csv',
    'base_de_dados/healthcare_fraud_detection.csv',        # Cópia no repositório: executando de dentro de PoC/
    'PoC/base_de_dados/healthcare_fraud_detection.csv',    # Executando da raiz do repositório
]
COLUNAS_CATEGORICAS_HC = ['Patient_Gender', 'Diagnosis_Code', 'Procedure_Code', 'Insurance_Type',
                          'Provider_Specialty', 'Patient_State', 'Claim_Status', 'Visit_Type']


def preparar_healthcare(df):
    # Mesma amostragem das outras bases (todas as fraudes + até 200 normais por fraude);
    # como esta base tem 8,3% de fraude, ela é mantida inteira.
    df = amostrar(df.dropna(subset=['Is_Fraud']), 'Is_Fraud')
    df = df.copy()
    df['Is_Fraud'] = df['Is_Fraud'].astype(int)
    # Identificadores não generalizam (como nameOrig/nameDest); a data é um carimbo de tempo (como Time)
    df = df.drop(columns=['Provider_ID', 'Claim_ID', 'Claim_Submission_Date'], errors='ignore')
    # Log dos valores (como amount_log) e proporção aprovada pela operadora (análoga ao errorBalance da PaySim)
    df['approved_ratio'] = df['Approved_Amount'] / df['Claim_Amount'].replace(0, np.nan)
    df['claim_amount_log'] = np.log1p(df['Claim_Amount'].clip(lower=0))
    df['approved_amount_log'] = np.log1p(df['Approved_Amount'].clip(lower=0))
    df = df.drop(columns=['Claim_Amount', 'Approved_Amount'])
    # Categóricas -> One-Hot (Procedure_Code é código, não quantidade); ausente vira 'Desconhecido'
    for col in COLUNAS_CATEGORICAS_HC:
        df[col] = df[col].astype('object').where(df[col].notna(), 'Desconhecido').astype(str)
    df = pd.get_dummies(df, columns=COLUNAS_CATEGORICAS_HC, drop_first=True, dtype=float)
    # Numéricas ausentes (Prior_Visits_12m) -> mediana
    for col in df.select_dtypes(include=np.number).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    return df, 'Is_Fraud'


PREPARADORES['healthcare'] = preparar_healthcare
DATASETS_HEALTHCARE = ['healthcare']
print("Base healthcare registrada:", CAMINHOS['healthcare'][0])


In [ ]:
# =====================================================================
# --- 5b. EXPERIMENTO NA BASE HEALTHCARE (mesma configuração da seção 5) ---
# =====================================================================
from sklearn.metrics import roc_curve

METRICA_PRINCIPAL = 'ROC-AUC'       # Métrica do teste de não-inferioridade ('ROC-AUC' ou 'PR-AUC')
MARGEM_NAO_INFERIORIDADE = 0.01     # Perda máxima aceitável do federado em relação ao centralizado
N_BOOTSTRAP = 300                   # Reamostragens do teste para o IC 95% do Δ médio

METRICAS = ['ROC-AUC', 'PR-AUC', 'Precisão', 'Recall', 'F1', 'Acurácia']
CENTRAL, FEDERADO, LOCAL = 'Centralizado', 'Federado + SMPC', 'Local isolado'
CONFIGS = [CENTRAL, FEDERADO, LOCAL]
CORES = {CENTRAL: '#1f77b4', FEDERADO: '#d62728', LOCAL: '#7f7f7f'}


def treinar_em_blocos(dados, X, y, semente, id_semente_bloco):
    """
    Treino SEM federação com o mesmo procedimento do FL: parte dos mesmos pesos iniciais, treina NUM_RODADAS blocos
    de EPOCAS_LOCAIS épocas e guarda o checkpoint com melhor PR-AUC na validação.
    """
    input_dim = dados['input_dim']
    seed_everything(semente)
    estado = novo_estado_global(input_dim)          # Mesma inicialização de executar_federado()
    historico, melhor, t_total = [], None, 0.0
    for bloco in range(1, NUM_RODADAS + 1):
        sincronizar()
        t0 = time.perf_counter()
        estado = treinar_local(estado, X, y, input_dim, semente * 1000 + bloco * 10 + id_semente_bloco)
        sincronizar()
        t_total += time.perf_counter() - t0
        pr_auc_val = average_precision_score(dados['y_val'], scores_anomalia(estado['D'], input_dim, dados['X_val']))
        historico.append(dict(rodada=bloco, pr_auc_val=pr_auc_val))
        if melhor is None or pr_auc_val > melhor[0]:
            melhor = (pr_auc_val, bloco, copy.deepcopy(estado))
    return melhor[2], melhor[1], pd.DataFrame(historico), t_total


def bootstrap_delta(y, pares_scores, n_boot, semente):
    """IC 95% do Δ médio entre sementes (federado − centralizado), reamostrando o conjunto de teste."""
    rng = np.random.default_rng(semente)
    deltas = {'ROC-AUC': [], 'PR-AUC': []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        for m, fn in [('ROC-AUC', roc_auc_score), ('PR-AUC', average_precision_score)]:
            deltas[m].append(np.mean([fn(y[idx], f[idx]) - fn(y[idx], c[idx]) for c, f in pares_scores]))
    return {m: np.array(v) for m, v in deltas.items()}


resultados_gerais = globals().get('resultados_gerais', {})  # preserva os resultados das outras bases

for nome_base in DATASETS_HEALTHCARE:
    print("\n" + "=" * 78)
    print(f" BASE: {nome_base.upper()} | {len(SEMENTES)} sementes | {NUM_RODADAS} x {EPOCAS_LOCAIS} épocas "
          f"| {len(DEGRAUS)} geradores")
    print("=" * 78)
    dados = carregar_dados(nome_base)
    if dados is None:
        print(f"Aviso: base '{nome_base}' não encontrada em {CAMINHOS[nome_base]}. Pulando.")
        continue
    X_central = np.concatenate([c[0] for c in dados['clientes']])   # União exata dos dados dos clientes
    y_central = np.concatenate([c[1] for c in dados['clientes']])

    linhas, tempos, pares_scores, curvas, exemplo = [], [], [], [], None
    for semente in SEMENTES:
        print(f"\n--- Semente {semente} ---")
        # (a) Centralizado
        est_c, bloco_c, hist_c, t_c = treinar_em_blocos(dados, X_central, y_central, semente, id_semente_bloco=9)
        print(f"[Centralizado] treino {t_c:.1f}s | melhor bloco {bloco_c} | PR-AUC val {hist_c['pr_auc_val'].max():.4f}")

        # (b) Federado + SMPC (modelo final)
        est_f, rodada_f, hist_f = executar_federado(dados, semente)
        t_f = hist_f['t_treino'].sum()

        # (c) Local isolado: cada cliente com as mesmas sementes que usa no federado
        avs_locais = []
        for k, (X_k, y_k) in enumerate(dados['clientes']):
            est_k, _, _, _ = treinar_em_blocos(dados, X_k, y_k, semente, id_semente_bloco=k)
            avs_locais.append(avaliar(est_k['D'], dados))
        print(f"[Local isolado] PR-AUC teste por cliente: {[round(a['PR-AUC'], 4) for a in avs_locais]}")

        av = {CENTRAL: avaliar(est_c['D'], dados), FEDERADO: avaliar(est_f['D'], dados),
              LOCAL: {m: np.mean([a[m] for a in avs_locais]) for m in METRICAS}}
        for cfg in CONFIGS:
            linhas.append({'Semente': semente, 'Configuração': cfg, **{m: av[cfg][m] for m in METRICAS}})
            print(f"  {cfg:16s} ROC-AUC {av[cfg]['ROC-AUC']:.4f} | PR-AUC {av[cfg]['PR-AUC']:.4f} | F1 {av[cfg]['F1']:.4f}")
        tempos.append({'Semente': semente, 'Centralizado': t_c, 'Federado (sequencial)': t_f,
                       'Federado (paralelo, estimado)': t_f / NUM_CLIENTES,
                       'SMPC total': hist_f['t_smpc'].sum(), 'Erro máx. SMPC': hist_f['erro_max'].max(),
                       'Bloco escolhido (central)': bloco_c, 'Rodada escolhida (federado)': rodada_f})
        pares_scores.append((av[CENTRAL]['scores'], av[FEDERADO]['scores']))
        curvas.append((hist_c.assign(semente=semente, cfg=CENTRAL), hist_f[['rodada', 'pr_auc_val']].assign(semente=semente, cfg=FEDERADO)))
        if exemplo is None:
            exemplo = {cfg: av[cfg]['scores'] for cfg in [CENTRAL, FEDERADO]}

    df = pd.DataFrame(linhas)
    df_tempos = pd.DataFrame(tempos).set_index('Semente')
    df_curvas = pd.concat([pd.concat(c) for c in curvas], ignore_index=True)

    # ----- Resumo -----
    resumo = df.groupby('Configuração')[METRICAS].agg(['mean', 'std']).reindex(CONFIGS)
    print(f"\n>> MÉTRICAS NO TESTE: média ± desvio-padrão em {len(SEMENTES)} sementes")
    print(pd.DataFrame({m: [f"{resumo.loc[c, (m, 'mean')]:.4f} ± {np.nan_to_num(resumo.loc[c, (m, 'std')]):.4f}"
                            for c in CONFIGS] for m in METRICAS}, index=CONFIGS).to_string())

    piv = df.pivot(index='Semente', columns='Configuração')
    deltas = {m: (piv[(m, FEDERADO)] - piv[(m, CENTRAL)]).values for m in METRICAS}
    print(f"\n>> Δ PAREADO POR SEMENTE ({FEDERADO} − {CENTRAL}):")
    for m in METRICAS:
        d = deltas[m]
        meia = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else 0.0
        print(f"  {m:9s}: {d.mean():+.4f}  (IC95% entre sementes: [{d.mean() - meia:+.4f}, {d.mean() + meia:+.4f}])")

    boot = bootstrap_delta(dados['y_test'], pares_scores, N_BOOTSTRAP, SEMENTES[0])
    ic = {m: np.percentile(v, [2.5, 97.5]) for m, v in boot.items()}
    print(f"\n>> IC 95% POR BOOTSTRAP DO Δ MÉDIO ({N_BOOTSTRAP} reamostragens do teste):")
    for m in ['ROC-AUC', 'PR-AUC']:
        veredito = 'não-inferior' if ic[m][0] > -MARGEM_NAO_INFERIORIDADE else 'não-inferioridade NÃO demonstrada'
        print(f"  Δ{m}: médio {deltas[m].mean():+.4f} | IC95% [{ic[m][0]:+.4f}, {ic[m][1]:+.4f}] -> {veredito} "
              f"(margem {MARGEM_NAO_INFERIORIDADE})")

    nao_inferior = ic[METRICA_PRINCIPAL][0] > -MARGEM_NAO_INFERIORIDADE
    print(f"\n>> RESPOSTA À QUESTÃO DE PESQUISA (métrica principal: {METRICA_PRINCIPAL}):")
    if nao_inferior:
        print(f"  O STEP-GAN Federado + SMPC é NÃO-INFERIOR ao baseline centralizado: limite inferior do IC95% "
              f"{ic[METRICA_PRINCIPAL][0]:+.4f} > {-MARGEM_NAO_INFERIORIDADE}.")
    else:
        print(f"  NÃO foi possível demonstrar não-inferioridade: limite inferior do IC95% "
              f"{ic[METRICA_PRINCIPAL][0]:+.4f} <= {-MARGEM_NAO_INFERIORIDADE}.")
    ganho = (piv[(METRICA_PRINCIPAL, FEDERADO)] - piv[(METRICA_PRINCIPAL, LOCAL)]).mean()
    print(f"  Ganho da colaboração ({FEDERADO} − {LOCAL}): {ganho:+.4f} {METRICA_PRINCIPAL}")
    print(f"  SMPC: erro máx. vs FedAvg em claro {df_tempos['Erro máx. SMPC'].max():.1e}")

    print("\n>> TEMPOS (s) E CHECKPOINTS ESCOLHIDOS POR SEMENTE:")
    print(df_tempos.to_string(formatters={'Erro máx. SMPC': lambda x: f"{x:.1e}"}, float_format=lambda x: f"{x:.2f}"))

    arquivo = f'resultados_teste6_{nome_base}.csv'
    df.to_csv(arquivo, index=False)
    print(f"Resultados por semente salvos em '{arquivo}'.")
    resultados_gerais[nome_base] = dict(metricas=df, tempos=df_tempos, ic_bootstrap=ic, nao_inferior=nao_inferior)

    # ----- Visualizações -----
    fig, axes = plt.subplots(2, 3, figsize=(22, 11))
    fig.suptitle(f"Teste 6 (v2): Centralizado × Federado + SMPC (STEP-GAN v5, {nome_base}, {len(SEMENTES)} sementes)",
                 fontsize=14, fontweight='bold')

    # 1. Métricas
    ax = axes[0, 0]
    largura = 0.8 / len(CONFIGS)
    x = np.arange(len(METRICAS))
    for i, cfg in enumerate(CONFIGS):
        ax.bar(x + (i - (len(CONFIGS) - 1) / 2) * largura, [resumo.loc[cfg, (m, 'mean')] for m in METRICAS], largura,
               yerr=[np.nan_to_num(resumo.loc[cfg, (m, 'std')]) for m in METRICAS], capsize=3,
               color=CORES[cfg], label=cfg)
    ax.set_xticks(x)
    ax.set_xticklabels(METRICAS)
    ax.set_ylim(0, 1.2)
    ax.set_title('Métricas no teste (média ± desvio)', fontweight='bold')
    ax.legend(loc='upper center', ncol=3)

    # 2. Curvas ROC (primeira semente)
    ax = axes[0, 1]
    for cfg in [CENTRAL, FEDERADO]:
        fpr, tpr, _ = roc_curve(dados['y_test'], exemplo[cfg])
        ax.plot(fpr, tpr, '-' if cfg == CENTRAL else '--', color=CORES[cfg],
                label=f"{cfg} (AUC={roc_auc_score(dados['y_test'], exemplo[cfg]):.4f})")
    ax.plot([0, 1], [0, 1], ':', color='gray')
    ax.set_title(f'Curva ROC no teste (semente {SEMENTES[0]})', fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(loc='lower right')

    # 3. Convergência: PR-AUC na validação por bloco/rodada
    ax = axes[0, 2]
    for cfg in [CENTRAL, FEDERADO]:
        sub = df_curvas[df_curvas['cfg'] == cfg]
        for _, g in sub.groupby('semente'):
            ax.plot(g['rodada'], g['pr_auc_val'], color=CORES[cfg], alpha=0.25)
        media = sub.groupby('rodada')['pr_auc_val'].mean()
        ax.plot(media.index, media.values, color=CORES[cfg], lw=2.5, marker='o', label=f'{cfg} (média)')
    ax.set_xlabel(f'Rodada / bloco de {EPOCAS_LOCAIS} épocas')
    ax.set_title('PR-AUC na validação', fontweight='bold')
    ax.legend()

    # 4. Δ por semente e IC de não-inferioridade
    ax = axes[1, 0]
    for i, m in enumerate(['ROC-AUC', 'PR-AUC', 'F1']):
        ax.scatter(np.full(len(deltas[m]), i), deltas[m], color=CORES[FEDERADO], zorder=3,
                   label='Δ por semente' if i == 0 else None)
        ax.plot([i - 0.2, i + 0.2], [deltas[m].mean()] * 2, color='black', lw=2, label='Δ médio' if i == 0 else None)
    for i, m in enumerate(['ROC-AUC', 'PR-AUC']):
        ax.errorbar(i + 0.3, np.mean(ic[m]), yerr=[[np.mean(ic[m]) - ic[m][0]], [ic[m][1] - np.mean(ic[m])]],
                    fmt='none', ecolor='#2ca02c', capsize=6, lw=2, label='IC95% bootstrap' if i == 0 else None)
    ax.axhline(0, color='gray', lw=1)
    ax.axhline(-MARGEM_NAO_INFERIORIDADE, color='#d62728', ls='--',
               label=f'Margem de não-inferioridade (−{MARGEM_NAO_INFERIORIDADE})')
    ax.set_xticks(range(3))
    ax.set_xticklabels(['ΔROC-AUC', 'ΔPR-AUC', 'ΔF1'])
    ax.set_title(f'{FEDERADO} − {CENTRAL}', fontweight='bold')
    ax.legend(loc='best', fontsize=9)

    # 5. Tempo de treino
    ax = axes[1, 1]
    media_t = df_tempos[['Centralizado', 'Federado (sequencial)', 'Federado (paralelo, estimado)']].mean()
    ax.bar(media_t.index, media_t.values, color=['#1f77b4', '#ff9896', '#d62728'])
    for i, v in enumerate(media_t.values):
        ax.text(i, v, f"{v:.1f}s", ha='center', va='bottom')
    ax.set_title(f"Tempo de treino (SMPC total: {df_tempos['SMPC total'].mean():.2f}s)", fontweight='bold')
    ax.set_ylabel('segundos')

    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.show()

print("\n[SUCESSO] Teste 6 (v2) concluído na base healthcare!")

In [ ]:
# =====================================================================
# --- COMPARATIVO ENTRE AS BASES (Teste 6: Centralizado × Federado + SMPC) ---
# =====================================================================
import contextlib
import io

NOMES_BASES = {'creditcard': 'Credit Card', 'paysim': 'PaySim', 'healthcare': 'Healthcare'}
# Cor fixa por base (paleta categórica validada para daltonismo): a mesma base tem sempre a mesma cor
CORES_BASES = {'creditcard': '#2a78d6', 'healthcare': '#eb6834', 'paysim': '#1baf7a'}
bases = [b for b in NOMES_BASES if b in globals().get('resultados_gerais', {})]


def perfil_da_base(nome):
    """Tamanho e taxa de fraude do conjunto de teste (a base é recarregada só para essas estatísticas)."""
    with contextlib.redirect_stdout(io.StringIO()):
        d = carregar_dados(nome)
    return {'Transações (teste)': len(d['y_test']), 'Taxa de fraude (teste)': float(np.mean(d['y_test'])),
            'Features': int(d['input_dim'])}

METRICAS_T6 = ['ROC-AUC', 'PR-AUC', 'F1']

if not bases:
    print("Nenhum resultado disponível: execute a seção 5 e/ou a base healthcare antes do comparativo.")
else:
    perfis = {b: perfil_da_base(b) for b in bases}
    linhas_metricas, linhas_delta = [], []
    for b in bases:
        metricas_b, ic_b = resultados_gerais[b]['metricas'], resultados_gerais[b]['ic_bootstrap']
        for cfg in CONFIGS:
            sub = metricas_b[metricas_b['Configuração'] == cfg]
            taxa = perfis[b]['Taxa de fraude (teste)']
            linhas_metricas.append({'Base': NOMES_BASES[b], 'Configuração': cfg,
                                    **{m: f"{sub[m].mean():.4f} ± {sub[m].std(ddof=1):.4f}" for m in METRICAS_T6},
                                    'PR-AUC normalizado': f"{((sub['PR-AUC'] - taxa) / (1 - taxa)).mean():.4f}"})
        piv = metricas_b.pivot(index='Semente', columns='Configuração')
        linha = {'Base': NOMES_BASES[b], 'Taxa de fraude (teste)': f"{perfis[b]['Taxa de fraude (teste)']:.2%}"}
        for m in ['ROC-AUC', 'PR-AUC']:
            delta = (piv[(m, FEDERADO)] - piv[(m, CENTRAL)]).mean()
            linha[f'Δ{m} (fed − central)'] = f"{delta:+.4f}"
            linha[f'IC95% Δ{m}'] = f"[{ic_b[m][0]:+.4f}; {ic_b[m][1]:+.4f}]"
            linha[f'Não-inferior em {m}?'] = 'sim' if ic_b[m][0] > -MARGEM_NAO_INFERIORIDADE else 'não demonstrado'
        linha['Ganho da colaboração (fed − local, ROC-AUC)'] = f"{(piv[('ROC-AUC', FEDERADO)] - piv[('ROC-AUC', LOCAL)]).mean():+.4f}"
        linhas_delta.append(linha)

    print(f">> MÉTRICAS NO TESTE POR BASE E CONFIGURAÇÃO ({len(SEMENTES)} sementes; média ± desvio-padrão)")
    print(pd.DataFrame(linhas_metricas).set_index(['Base', 'Configuração']).to_string())
    print(f"\n>> NÃO-INFERIORIDADE DO FEDERADO + SMPC POR BASE (margem {MARGEM_NAO_INFERIORIDADE}; "
          f"métrica principal: {METRICA_PRINCIPAL})")
    print(pd.DataFrame(linhas_delta).set_index('Base').T.to_string())
    print("\nObservação: o ROC-AUC não depende da taxa de fraude e é a métrica mais direta para comparar as bases. "
          "O PR-AUC depende dela (um classificador aleatório tem PR-AUC igual à taxa de fraude); por isso a comparação "
          "do PR-AUC entre bases deve usar o PR-AUC normalizado = (PR-AUC − taxa) / (1 − taxa): 0 = aleatório, 1 = perfeito.")

    fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
    largura = 0.8 / len(bases)
    x = np.arange(len(CONFIGS))
    for eixo, m in zip(axes[:2], ['ROC-AUC', 'PR-AUC']):
        for i, b in enumerate(bases):
            metricas_b = resultados_gerais[b]['metricas']
            medias = [metricas_b.loc[metricas_b['Configuração'] == c, m].mean() for c in CONFIGS]
            desvios = [metricas_b.loc[metricas_b['Configuração'] == c, m].std(ddof=1) for c in CONFIGS]
            eixo.bar(x + (i - (len(bases) - 1) / 2) * largura, medias, largura, yerr=desvios, capsize=3,
                     color=CORES_BASES[b], label=NOMES_BASES[b], edgecolor='white', linewidth=1.5)
        eixo.set_xticks(x)
        eixo.set_xticklabels(CONFIGS)
        eixo.set_ylim(0, 1.2)
        eixo.set_title(f'{m} no teste por configuração', fontweight='bold')
        eixo.legend(loc='upper center', ncol=len(bases), frameon=False)

    for i, b in enumerate(bases):
        piv = resultados_gerais[b]['metricas'].pivot(index='Semente', columns='Configuração')
        ic_b = resultados_gerais[b]['ic_bootstrap']
        for j, m in enumerate(['ROC-AUC', 'PR-AUC']):
            centro = (piv[(m, FEDERADO)] - piv[(m, CENTRAL)]).mean()
            posicao = j + (i - (len(bases) - 1) / 2) * 0.25
            axes[2].errorbar(posicao, centro, yerr=[[centro - ic_b[m][0]], [ic_b[m][1] - centro]], fmt='o', capsize=6,
                             color=CORES_BASES[b], label=NOMES_BASES[b] if j == 0 else None)
    axes[2].axhline(0, color='gray', lw=1)
    axes[2].axhline(-MARGEM_NAO_INFERIORIDADE, color='#52514e', ls='--', label=f'Margem (−{MARGEM_NAO_INFERIORIDADE})')
    axes[2].set_xticks([0, 1])
    axes[2].set_xticklabels(['ΔROC-AUC', 'ΔPR-AUC'])
    axes[2].set_title('Federado + SMPC − Centralizado (IC95% bootstrap)', fontweight='bold')
    axes[2].legend(fontsize=9)
    plt.suptitle('Teste 6 — Centralizado × Federado + SMPC: comparativo entre as bases', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
